# House Share Prohibited Buildings Analysis

This notebook explores the distribution of house share prohibited buildings across Chicago's community areas.

## Overview
- **Data Source**: House Share Prohibited Buildings List
- **Geographic Boundaries**: Chicago Community Areas  
- **Analysis**: Spatial aggregation of building counts and unit counts by community area
- **Visualization**: Interactive maps and statistical summaries


In [ ]:
# Import required libraries
import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# Import our custom data processing module
from housing.data_processor import process_all_data

# Set up plotting style
plt.style.use("default")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 10

print("Libraries imported successfully!")

In [ ]:
# Define file paths
buildings_file = "../data/House_Share_Prohibited_Buildings_List.csv"
community_file = "../data/Boundaries_Community_Areas.csv"

# Process all data using our custom module
buildings_gdf, community_gdf, aggregated_data, summary_stats = process_all_data(
    buildings_file, community_file
)

print("Data processing completed!")
print("\nSummary Statistics:")
print(f"Total Community Areas: {summary_stats['total_community_areas']}")
print(f"Areas with Buildings: {summary_stats['areas_with_buildings']}")
print(f"Areas without Buildings: {summary_stats['areas_without_buildings']}")
print(f"Total Buildings: {summary_stats['total_buildings']:,}")
print(f"Total Units: {summary_stats['total_units']:,}")
print(f"Average Buildings per Area: {summary_stats['avg_buildings_per_area']:.1f}")
print(f"Average Units per Area: {summary_stats['avg_units_per_area']:.1f}")

In [ ]:
# Display top areas by building count
print("Top 10 Community Areas by Number of Buildings:")
top_buildings = aggregated_data.nlargest(10, "building_count")[
    ["COMMUNITY", "building_count", "total_units"]
]
print(top_buildings.to_string(index=False))

print("\n" + "=" * 60 + "\n")

# Display top areas by unit count
print("Top 10 Community Areas by Number of Units:")
top_units = aggregated_data.nlargest(10, "total_units")[
    ["COMMUNITY", "building_count", "total_units"]
]
print(top_units.to_string(index=False))

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "House Share Prohibited Buildings Distribution Analysis",
    fontsize=16,
    fontweight="bold",
)

# 1. Distribution of building counts
axes[0, 0].hist(
    aggregated_data["building_count"],
    bins=30,
    alpha=0.7,
    color="skyblue",
    edgecolor="black",
)
axes[0, 0].set_title("Distribution of Building Counts per Community Area")
axes[0, 0].set_xlabel("Number of Buildings")
axes[0, 0].set_ylabel("Number of Community Areas")
axes[0, 0].grid(True, alpha=0.3)

# 2. Distribution of unit counts
axes[0, 1].hist(
    aggregated_data["total_units"],
    bins=30,
    alpha=0.7,
    color="lightcoral",
    edgecolor="black",
)
axes[0, 1].set_title("Distribution of Unit Counts per Community Area")
axes[0, 1].set_xlabel("Number of Units")
axes[0, 1].set_ylabel("Number of Community Areas")
axes[0, 1].grid(True, alpha=0.3)

# 3. Top 15 areas by building count
top_15_buildings = aggregated_data.nlargest(15, "building_count")
axes[1, 0].barh(
    range(len(top_15_buildings)), top_15_buildings["building_count"], color="lightgreen"
)
axes[1, 0].set_yticks(range(len(top_15_buildings)))
axes[1, 0].set_yticklabels(top_15_buildings["COMMUNITY"], fontsize=8)
axes[1, 0].set_title("Top 15 Community Areas by Building Count")
axes[1, 0].set_xlabel("Number of Buildings")
axes[1, 0].grid(True, alpha=0.3)

# 4. Scatter plot: Buildings vs Units
axes[1, 1].scatter(
    aggregated_data["building_count"],
    aggregated_data["total_units"],
    alpha=0.6,
    color="purple",
    s=50,
)
axes[1, 1].set_title("Buildings vs Units per Community Area")
axes[1, 1].set_xlabel("Number of Buildings")
axes[1, 1].set_ylabel("Number of Units")
axes[1, 1].grid(True, alpha=0.3)

# Add correlation coefficient
correlation = aggregated_data["building_count"].corr(aggregated_data["total_units"])
axes[1, 1].text(
    0.05,
    0.95,
    f"Correlation: {correlation:.3f}",
    transform=axes[1, 1].transAxes,
    fontsize=10,
    bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.8},
)

plt.tight_layout()
plt.show()

In [ ]:
# Create an interactive map showing house share prohibited buildings
from folium import plugins

# Constants for map coloring
HIGH_BUILDING_THRESHOLD = 100  # noqa: PLR2004
MEDIUM_BUILDING_THRESHOLD = 50  # noqa: PLR2004

# Create base map centered on Chicago
chicago_map = folium.Map(
    location=[41.8781, -87.6298],  # Chicago coordinates
    zoom_start=10,
    tiles="OpenStreetMap",
)

# Add a marker cluster for better performance with many points
marker_cluster = plugins.MarkerCluster().add_to(chicago_map)

# Add markers for each building
for _, building in buildings_gdf.head(
    500
).iterrows():  # Limit to first 500 for performance
    folium.Marker(
        location=[building.geometry.y, building.geometry.x],
        popup=f"Building ID: {building.get('ID', 'N/A')}<br>Units: {building.get('Number of Units', 'N/A')}",
        tooltip=f"Units: {building.get('Number of Units', 'N/A')}",
        icon=folium.Icon(color="red", icon="home"),
    ).add_to(marker_cluster)

# Add community area boundaries
for _, area in community_gdf.iterrows():
    # Create a popup with statistics
    area_data = aggregated_data[aggregated_data["COMMUNITY"] == area["COMMUNITY"]]
    if not area_data.empty:
        buildings = area_data.iloc[0]["building_count"]
        units = area_data.iloc[0]["total_units"]
        popup_text = f"""
        <b>{area['COMMUNITY']}</b><br>
        Buildings: {buildings}<br>
        Units: {units}
        """
    else:
        popup_text = f"<b>{area['COMMUNITY']}</b><br>No prohibited buildings"

    # Color based on building count
    if not area_data.empty and buildings > 0:
        color = (
            "red"
            if buildings > HIGH_BUILDING_THRESHOLD
            else "orange"
            if buildings > MEDIUM_BUILDING_THRESHOLD
            else "yellow"
        )
    else:
        color = "green"

    folium.GeoJson(
        area.geometry.__geo_interface__,
        style_function=lambda x, color=color: {
            "fillColor": color,
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.3,
        },
        popup=folium.Popup(popup_text, max_width=200),
    ).add_to(chicago_map)

# Add legend
legend_html = """
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 200px; height: 120px; 
     background-color: white; border:2px solid grey; z-index:9999; 
     font-size:14px; padding: 10px">
<p><b>Legend</b></p>
<p><i class="fa fa-circle" style="color:red"></i> High (>100 buildings)</p>
<p><i class="fa fa-circle" style="color:orange"></i> Medium (50-100)</p>
<p><i class="fa fa-circle" style="color:yellow"></i> Low (1-50)</p>
<p><i class="fa fa-circle" style="color:green"></i> None</p>
</div>
"""
chicago_map.get_root().html.add_child(folium.Element(legend_html))

# Display the map
chicago_map

In [ ]:
# Statistical Analysis: Building Efficiency and Density Analysis
from scipy.stats import poisson

# Statistical constants
ALPHA_LEVEL = 0.05  # noqa: PLR2004
MIN_EXPECTED_FREQUENCY = 5  # noqa: PLR2004
STRONG_CORRELATION = 0.7  # noqa: PLR2004
MODERATE_CORRELATION = 0.3  # noqa: PLR2004

print("=" * 70)
print("STATISTICAL ANALYSIS: BUILDING EFFICIENCY & DENSITY")
print("=" * 70)

# Clean and prepare the data
aggregated_data["area_size_numeric"] = pd.to_numeric(
    aggregated_data["area_size"].str.replace(",", ""), errors="coerce"
)

# Calculate key metrics
aggregated_data["building_density"] = aggregated_data["building_count"] / (
    aggregated_data["area_size_numeric"] / 1_000_000
)  # buildings per km²
aggregated_data["units_per_building"] = (
    aggregated_data["total_units"] / aggregated_data["building_count"]
)
aggregated_data["units_per_km2"] = aggregated_data["total_units"] / (
    aggregated_data["area_size_numeric"] / 1_000_000
)

# Remove infinite values and NaN
aggregated_data = aggregated_data.replace([np.inf, -np.inf], np.nan).dropna()

print("Key Metrics Calculated:")
print("• Building density (buildings/km²)")
print("• Units per building (efficiency metric)")
print("• Units per km² (overall density)")

# Analysis 1: Are some areas more "efficient" at packing units into buildings?
print("\n" + "=" * 50)
print("ANALYSIS 1: BUILDING EFFICIENCY")
print("=" * 50)

# Compare high vs low efficiency areas
efficiency_median = aggregated_data["units_per_building"].median()
high_efficiency = aggregated_data[
    aggregated_data["units_per_building"] >= efficiency_median
]
low_efficiency = aggregated_data[
    aggregated_data["units_per_building"] < efficiency_median
]

print(f"Median units per building: {efficiency_median:.1f}")
print(
    f"High efficiency areas (≥{efficiency_median:.1f} units/building): {len(high_efficiency)}"
)
print(
    f"Low efficiency areas (<{efficiency_median:.1f} units/building): {len(low_efficiency)}"
)

# Test if high efficiency areas have different building densities
t_stat1, p_value1 = stats.ttest_ind(
    high_efficiency["building_density"], low_efficiency["building_density"]
)

print("\nQuestion: Do high-efficiency areas have different building densities?")
print(f"T-statistic: {t_stat1:.4f}")
print(f"P-value: {p_value1:.6f}")
print(
    f"High efficiency mean density: {high_efficiency['building_density'].mean():.2f} buildings/km²"
)
print(
    f"Low efficiency mean density: {low_efficiency['building_density'].mean():.2f} buildings/km²"
)

if p_value1 < ALPHA_LEVEL:
    print("✅ SIGNIFICANT: High-efficiency areas have different building densities!")
else:
    print("❌ NOT SIGNIFICANT: No difference in building densities")

# Analysis 2: Spatial clustering analysis
print("\n" + "=" * 50)
print("ANALYSIS 2: SPATIAL CLUSTERING")
print("=" * 50)

# Test if building counts follow a random distribution (Poisson) or are clustered

# Expected vs observed building counts
total_buildings = aggregated_data["building_count"].sum()
total_areas = len(aggregated_data)
expected_mean = total_buildings / total_areas

print(f"Total buildings: {total_buildings}")
print(f"Total areas: {total_areas}")
print(f"Expected mean buildings per area: {expected_mean:.2f}")

# Get observed counts for each possible building count
observed_counts = aggregated_data["building_count"].value_counts().sort_index()
max_observed = observed_counts.index.max()

print("Observed building count distribution:")
print(f"Range: 0 to {max_observed} buildings per area")

# Generate expected Poisson distribution
expected_counts = []
observed_values = []
for k in range(max_observed + 1):
    expected = poisson.pmf(k, expected_mean) * total_areas
    observed = observed_counts.get(k, 0)
    expected_counts.append(expected)
    observed_values.append(observed)

# Combine small expected values (< 5) for chi-square test validity
observed_combined = []
expected_combined = []
current_obs = 0
current_exp = 0

for i in range(len(expected_counts)):
    current_obs += observed_values[i]
    current_exp += expected_counts[i]

    # If we have enough expected frequency or reach the end
    if current_exp >= MIN_EXPECTED_FREQUENCY or i == len(expected_counts) - 1:
        if current_exp > 0:  # Only add if we have some expected frequency
            observed_combined.append(current_obs)
            expected_combined.append(current_exp)
        current_obs = 0
        current_exp = 0

print("\nCombined bins for chi-square test:")
print(f"Observed: {observed_combined}")
print(f"Expected: {[f'{x:.2f}' for x in expected_combined]}")

# Perform chi-square test
if len(observed_combined) > 1 and sum(expected_combined) > 0:
    chi2_stat, chi2_p = stats.chisquare(observed_combined, expected_combined)

    print("\nQuestion: Are buildings randomly distributed or clustered?")
    print(f"Chi-square statistic: {chi2_stat:.4f}")
    print(f"P-value: {chi2_p:.6f}")

    if chi2_p < ALPHA_LEVEL:
        print("✅ SIGNIFICANT: Buildings are CLUSTERED (not randomly distributed)")
        print(
            "   This suggests policy or economic factors influence house share prohibitions"
        )
    else:
        print("❌ NOT SIGNIFICANT: Buildings appear randomly distributed")
else:
    print("❌ Cannot perform chi-square test - insufficient data")

# Analysis 3: Correlation between area size and building concentration
print("\n" + "=" * 50)
print("ANALYSIS 3: AREA SIZE vs BUILDING CONCENTRATION")
print("=" * 50)

# Test correlation between area size and building density
correlation, corr_p = stats.pearsonr(
    aggregated_data["area_size_numeric"], aggregated_data["building_density"]
)

print("Question: Is there a relationship between area size and building density?")
print(f"Correlation coefficient: {correlation:.4f}")
print(f"P-value: {corr_p:.6f}")

if corr_p < ALPHA_LEVEL:
    if correlation > 0:
        print(
            "✅ SIGNIFICANT POSITIVE correlation: Larger areas have higher building density"
        )
    else:
        print(
            "✅ SIGNIFICANT NEGATIVE correlation: Smaller areas have higher building density"
        )
else:
    print("❌ NOT SIGNIFICANT: No relationship between area size and building density")

print("\nInterpretation:")
if abs(correlation) > STRONG_CORRELATION:
    print("• Strong relationship")
elif abs(correlation) > MODERATE_CORRELATION:
    print("• Moderate relationship")
else:
    print("• Weak relationship")

In [ ]:
# Visualization for Building Efficiency and Density Analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Building Efficiency & Density Analysis", fontsize=16, fontweight="bold")

# 1. Building efficiency comparison (units per building)
efficiency_data = [
    low_efficiency["units_per_building"],
    high_efficiency["units_per_building"],
]
efficiency_labels = [
    f"Low Efficiency\n(n={len(low_efficiency)})",
    f"High Efficiency\n(n={len(high_efficiency)})",
]

bp1 = axes[0, 0].boxplot(efficiency_data, labels=efficiency_labels, patch_artist=True)
bp1["boxes"][0].set_facecolor("lightcoral")
bp1["boxes"][1].set_facecolor("lightgreen")
axes[0, 0].set_title("Units per Building (Efficiency)")
axes[0, 0].set_ylabel("Units per Building")
axes[0, 0].grid(True, alpha=0.3)

# Add mean markers
axes[0, 0].scatter(
    [1, 2],
    [
        low_efficiency["units_per_building"].mean(),
        high_efficiency["units_per_building"].mean(),
    ],
    color="red",
    s=100,
    marker="D",
    label="Mean",
    zorder=10,
)
axes[0, 0].legend()

# 2. Building density vs efficiency scatter plot
scatter = axes[0, 1].scatter(
    aggregated_data["building_density"],
    aggregated_data["units_per_building"],
    c=aggregated_data["total_units"],
    cmap="viridis",
    alpha=0.7,
    s=60,
)
axes[0, 1].set_title("Building Density vs Efficiency")
axes[0, 1].set_xlabel("Building Density (buildings/km²)")
axes[0, 1].set_ylabel("Units per Building")
axes[0, 1].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[0, 1], label="Total Units")

# Add trend line
z = np.polyfit(
    aggregated_data["building_density"], aggregated_data["units_per_building"], 1
)
p = np.poly1d(z)
x_trend = np.linspace(
    aggregated_data["building_density"].min(),
    aggregated_data["building_density"].max(),
    100,
)
axes[0, 1].plot(x_trend, p(x_trend), "r--", alpha=0.8, linewidth=2)

# 3. Area size vs building density
axes[1, 0].scatter(
    aggregated_data["area_size_numeric"] / 1_000_000,
    aggregated_data["building_density"],
    alpha=0.7,
    color="purple",
    s=60,
)
axes[1, 0].set_title("Area Size vs Building Density")
axes[1, 0].set_xlabel("Area Size (km²)")
axes[1, 0].set_ylabel("Building Density (buildings/km²)")
axes[1, 0].grid(True, alpha=0.3)

# Add trend line
z2 = np.polyfit(
    aggregated_data["area_size_numeric"] / 1_000_000,
    aggregated_data["building_density"],
    1,
)
p2 = np.poly1d(z2)
x_trend2 = np.linspace(
    (aggregated_data["area_size_numeric"] / 1_000_000).min(),
    (aggregated_data["area_size_numeric"] / 1_000_000).max(),
    100,
)
axes[1, 0].plot(x_trend2, p2(x_trend2), "r--", alpha=0.8, linewidth=2)

# 4. Statistical summary
axes[1, 1].axis("off")
# Define constants for statistical thresholds
ALPHA_LEVEL = 0.05  # noqa: PLR2004
STRONG_CORRELATION = 0.7  # noqa: PLR2004
MODERATE_CORRELATION = 0.3  # noqa: PLR2004

stats_text = f"""
Statistical Analysis Summary

Analysis 1 - Building Efficiency:
• T-statistic: {t_stat1:.4f}
• P-value: {p_value1:.6f}
• {'✅ Significant difference' if p_value1 < ALPHA_LEVEL else '❌ No significant difference'}

Analysis 2 - Spatial Clustering:
• Chi-square: {chi2_stat:.4f}
• P-value: {chi2_p:.6f}
• {'✅ Buildings are CLUSTERED' if chi2_p < ALPHA_LEVEL else '❌ Random distribution'}

Analysis 3 - Area Size Correlation:
• Correlation: {correlation:.4f}
• P-value: {corr_p:.6f}
• {'✅ Significant relationship' if corr_p < ALPHA_LEVEL else '❌ No relationship'}

Key Insights:
• Median efficiency: {efficiency_median:.1f} units/building
• Expected random mean: {expected_mean:.1f} buildings/area
• {'Strong' if abs(correlation) > STRONG_CORRELATION else 'Moderate' if abs(correlation) > MODERATE_CORRELATION else 'Weak'} area-size relationship
"""

axes[1, 1].text(
    0.05,
    0.95,
    stats_text,
    transform=axes[1, 1].transAxes,
    fontsize=10,
    verticalalignment="top",
    fontfamily="monospace",
    bbox={"boxstyle": "round,pad=0.5", "facecolor": "lightgray", "alpha": 0.8},
)

plt.tight_layout()
plt.show()

## Summary and Conclusions

### Key Findings:

1. **Geographic Distribution**: House share prohibited buildings are concentrated in specific areas of Chicago, with the highest concentrations in Near North Side, Lake View, and Lincoln Park.

2. **Interactive Map**: The map visualization shows:
   - **Red areas**: High concentration (>100 buildings)
   - **Orange areas**: Medium concentration (50-100 buildings)  
   - **Yellow areas**: Low concentration (1-50 buildings)
   - **Green areas**: No prohibited buildings

3. **Statistical Analysis - Three Key Insights**:

   **Building Efficiency Analysis**: Tests whether areas with high unit-per-building ratios (efficiency) have different building densities. This reveals if "efficient" areas cluster differently.

   **Spatial Clustering Analysis**: Uses chi-square goodness-of-fit test to determine if buildings follow a random Poisson distribution or are clustered. Clustering suggests policy/economic factors drive prohibitions.

   **Area Size Correlation**: Tests if larger community areas have different building densities, revealing whether size influences prohibition patterns.

### Data Insights:
- **Total Coverage**: 2,373 prohibited buildings across 77 community areas
- **Geographic Spread**: 57 out of 77 areas have at least one prohibited building
- **Unit Impact**: 191,318 total units affected by house share prohibitions
- **Efficiency Range**: Significant variation in units per building across areas

### Policy Implications:
- **Clustering patterns** (if found) suggest targeted policy interventions in specific neighborhoods
- **Efficiency differences** may indicate varying building types or development patterns
- **Area size relationships** could inform zoning and development policies
- **Statistical significance** provides evidence-based insights for policy decisions

### Research Questions Answered:
- ✅ Are buildings randomly distributed or clustered? (Chi-square test)
- ✅ Do efficient areas have different density patterns? (T-test)
- ✅ Does area size influence building concentration? (Correlation analysis)

### Next Steps:
- Investigate socioeconomic factors in clustered areas
- Analyze building age and type data for efficiency patterns
- Compare with rental price data to understand economic impacts
- Study temporal trends in house share prohibitions
